In [ ]:
1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
 - 토큰수 초과로 답변을 생성하지 못할 수 있고
 - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을때, 벡터 db 유사도 검색
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달

In [2]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade --quiet docx2txt langchain-community
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#1. 문서의 내용을 읽는다
#2. 문서를 쪼갠다
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200,
)
loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

document_list
len(document_list)
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings




[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/var/folders/jw/qym3n0892n51wr3xcmd925d00000gn/T/ipykernel_82177/230340770.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [4]:
#3. 임베딩 -> 벡터 데이터베이스에 저장
#4. 질문이 있을때, 벡터 db 유사도 검색
%pip install langchain-chroma
load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

from langchain_chroma import Chroma

database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory='./chroma')

query = '연봉 5천만원인 직장인의 소득세는 얼마?'
retrieved_docs = database.similarity_search(query, k=3)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
query = '연봉 5천만원인 직장인의 소득세는 얼마?'
retrieved_docs = database.similarity_search(query, k=3)
retrieved_docs

[Document(id='9d105457-6083-44c6-a0f4-e489db6e0413', metadata={'source': './tax.docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015.

In [8]:
#5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

prompt = """[Identity]
-당신은 최고의 한국 소득세 전문가 입니다.
-[Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question : {query}
"""

ai_message = llm.invoke(prompt)

ai_message.content


'질문에 대한 답변을 드리겠습니다.\n\n[Context]에 포함된 내용이 없다면, 일반적인 정보를 제공해 드리겠습니다. 소득세는 개인의 소득에 대해 부과되는 세금으로, 소득의 종류와 금액에 따라 세율이 달라집니다. 주요 소득세 종류로는 근로소득세, 사업소득세, 이자소득세, 배당소득세, 양도소득세 등이 있습니다. \n\n세율과 관련하여, 2022년 현재 대한민국의 소득세율은 다음과 같습니다.\n\n- 1,200만원 이하: 6%\n- 1,200만원 초과 4,600만원 이하: 15%\n- 4,600만원 초과 8,800만원 이하: 24%\n- 8,800만원 초과 1억5천만원 이하: 35%\n- 1억5천만원 초과 3억원 이하: 38%\n- 3억원 초과 5억원 이하: 40%\n- 5억원 초과: 42%\n\n추가적으로, 소득공제 및 세액공제 항목에 따라 실제 납부해야 할 세액이 달라질 수 있습니다. 더 자세한 정보나 개별적인 상황에 맞는 상담이 필요하시다면, 공식적인 세무 상담 채널을 통해 확인하시는 것이 좋습니다.\n\n질문이 있으시면 더 자세히 답변해 드리겠습니다.'

In [ ]:
#위 방식 말고 좀 더 효율적으로 작성도 가능한데..
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달

%pip install langchainhub --quiet
from langchainhub import Client
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(), #이렇게 하면 사용 Db가 바뀌더라도 유연하게 대응가능
    chain_type_kwargs={"prompt":prompt}
)

ai_message = qa_chain({"query":query})
ai_message


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
